# Transform Circuits Data

1. Read bronze `circuits` table
1. Keep only the columns required for analytics (Drop `url` column)
1. Standardise column names using snake_case (`circuitId` → `circuit_id`, `circuitName` → `circuit_name`)
1. Rename columns to make them more meaningful (`lat` → `latitude`, `long` → `longitude`)
1. Filter out rows where `circuit_id` is null (business key validation)
1. Remove duplicate records
1. Transform values of columns `circuit_name` and `locality` to Title Case
1. Write the transformed data to silver `circuits` table


#### Entity Relationship Diagram - Formula1 Schema

![Formula1 Raw Data.png](../../z-course-images/formula1-raw-data-erd.png "Formula1 Raw Data.png")



In [0]:
%run "../00-common/01.environment-config"


In [0]:
val bronze_table = catalog_name + "." + bronze_schema + "." + "circuits"
val silver_table = catalog_name + "." + silver_schema + "." + "circuits"

#### Step 1 - Read bronze `circuits` table

In [0]:
val circuits_df = spark.read.option("versionAsOf","0").table(bronze_table)

In [0]:
val circuits_df= spark.table(bronze_table)

#### Step 2 - Keep only the columns required for analytics (Drop url column)

In [0]:
val circuits_selected_df = circuits_df.select(
  "circuitId","circuitName","lat","long","locality","country","ingestion_timestamp","source_file"
)

#### Step 3 & 4 - Standardise Column Names
- Standardise column names using snake_case (`circuitId` → `circuit_id`, `circuitName` → `circuit_name`)
- Rename columns to make them more meaningful (`lat` → `latitude`, `long` → `longitude`)


In [0]:
val circuits_renamed_df = circuits_selected_df
.withColumnRenamed("circuitId","circuit_id")
.withColumnRenamed("circuitName", "circuit_name")
.withColumnRenamed("lat", "latitude")
.withColumnRenamed("long", "longitude")

#### Step 5 - Filter out rows where circuit_id is null (business key validation)

In [0]:
import org.apache.spark.sql.functions.col
val circuits_valid_df= circuits_renamed_df.filter(col("circuit_id").isNotNull)

#### Step 6 - Remove duplicate records

In [0]:
//val circuits_distinct_df = circuits_valid_df.distinct()
val circuits_distinct_df = circuits_valid_df.dropDuplicates("circuit_id")

In [0]:
display(circuits_distinct_df)

#### Step 7 - Transform values of columns `circuit_name` and `locality` to Title Case


In [0]:
import org.apache.spark.sql.functions.{initcap,col}
val circuits_final_df= circuits_distinct_df
.withColumn("circuit_name",initcap(col("circuit_name")))
.withColumn("locality",initcap(col("locality")))

In [0]:
display(circuits_final_df)

#### Step 8 - Write the transformed data to silver `circuits` table

In [0]:
circuits_final_df.write.format("delta").mode("overwrite")
        .saveAsTable(silver_table)


In [0]:
display(spark.table(silver_table))